# D2.1 · Agent-assisted reconstruction

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *AI for Security*

Builds on **[D1.11 · Honeypots, canaries and deception in the agent's environment](https://spbreed.github.io/cyber-commons/lessons/D1.11.html)**.

| | |
|---|---|
| Tools used | Velociraptor, OpenSearch, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Reconstruct a timeline from raw logs with a context-loaded agent.

**Why a security engineer needs it.** Reaching for the agent once you're already behind. The control it builds is: pre-load logs, telemetry, segmentation model and playbooks.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Reconstruction is reading, and agents read fast. The speed is real and so is the failure mode: a timeline that is 95% right and completely confident is worse than no timeline, because somebody will make decisions on it.

> **At CyberTravels.** Reconstructing what CyberTravels did across six log sources is reading, and agents read fast. A timeline that is 95% right and fully confident is worse than none.

## 2 · The framework

```
   scattered evidence            reconstructed timeline
   +------------------+          +---------------------+
   | 6 log sources    |   -->    | ordered, attributed |
   | 900k lines       |          | 40 events           |
   +------------------+          +---------------------+
                                          |
                              every claim carries its source line
                              unsourced claim -> not in the timeline
```

Reconstruction is the first phase of any incident: build the timeline, establish
what happened, decide what to contain.

An agent makes this faster and more dangerous at the same time. Faster, because
a model can correlate thousands of log lines in seconds. More dangerous, because
it will produce a fluent, confident narrative from logs that were never
sufficient to support one — and a fluent narrative is much harder to challenge
than an obviously incomplete one.

So the discipline is to separate two questions that feel like one:

1. **What do the logs say?**
2. **What can the logs support?**

The gap between them is where reconstruction goes wrong, and it is the responder's
job to state that gap explicitly in the incident record.

## 3 · The control — state what the evidence can support

The fix is not a better model. It is a reconstruction step that reports its own evidentiary limits before it reports a conclusion.

## 4 · The procedure, as a skill

The timeline attributes every action to `dana@corp` and the fluent narrative recommends suspending her. The skill renders that view first, produces the truth view from an independent source, and then gates every claim on a field that supports it.

### The skill — [`skills/response/incident-reconstruction-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/incident-reconstruction-check/SKILL.md)

```yaml
name: incident-reconstruction-check
description: >-
  Reconstruct an incident from logs that attribute every action to a human,
  check what the evidence actually supports, and refuse to publish a narrative
  it does not carry. Use during response, when the timeline names a person and
  an agent did the work.
allowed-tools: Read, Grep, Glob
```

# The fluent narrative recommends suspending the wrong person

Agent logs that record only the delegated principal produce a timeline in which
a human did everything. A model asked to summarise it writes something fluent
and confident that recommends suspending her. The reconstruction is not wrong
about the events; it is wrong about the actor, and nothing in the log says so.

## When to use this

Every incident involving an agent, and before any narrative reaches a person who
will act on it.

## Procedure

**1 — Render the timeline as the logs have it.** Do not correct it yet. This is
what an investigator would see, and seeing it is the point.

**2 — Ask which field carries the acting identity.** Not the delegated
principal — the identity that performed the action. If no field carries it, stop:
every attribution below is inherited from the request, not observed.

**3 — Produce the truth view from an independent source** where one exists — host
accounting, the gateway, the downstream's own log. Diff it against the timeline
and record what changes. Usually one or two actions move from the human to the
agent, and they are the important ones.

**4 — Gate the narrative on evidence.** A summary may assert only what a field
supports. Attach the field to each claim; a claim with no field is removed, not
softened.

**5 — Publish the safe version and the gap.** State plainly which questions the
record cannot answer. An investigation that says so is more useful than one that
fills the gap fluently.

## Example

**Input** — the fixture committed at the top of [`scripts/incident_reconstruction_check.py`](scripts/incident_reconstruction_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
WHAT THE RESPONDER SEES
   t+s  actor           action          target
     0  dana@corp       login           sso
    22  dana@corp       open_ticket     SEC-4471
    40  dana@corp       read_file       /work/repo/billing.py
    41  dana@corp       read_file       /home/app/.aws/credentials
    43  dana@corp       http_post       collect.example.com
   180  dana@corp       logout          sso
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "timeline": [{"at": "str", "attributed_to": "str", "action": "str"}],
  "acting_identity_field": "str|null",
  "truth_view": [{"at": "str", "actual_actor": "str", "action": "str", "source": "str"}],
  "narrative": {"claims": [{"text": "str", "supported_by": "str|null"}], "removed": 0},
  "gaps": ["str"]
}
```

## Failure modes

- **Publishing the fluent version.** It is confident and it names a person.
- **Correcting the timeline before showing it.** The uncorrected view is what
  everyone else is looking at.
- **Softening unsupported claims.** Remove them.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/incident-reconstruction-check/scripts/incident_reconstruction_check.py
SCRIPT = "skills/response/incident-reconstruction-check/scripts/incident_reconstruction_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The timeline attributes every action to `dana@corp`. The fluent narrative recommends suspending her. The truth view shows `patch-agent` performed the credential read and the external POST; reconstruction reports BROKEN attribution with 3 misattributed lines. The evidence check flags the missing acting-identity field, the missing chain, and a non-human action rate.

## Your turn

Take a real incident timeline from your own history and ask what it would look like if an agent had been operating on the user's credential. If you cannot tell from the logs, your reconstructions already carry this risk.

---

**Next → [D2.2 · When the actor is an agent](https://spbreed.github.io/cyber-commons/lessons/D2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*